# Auto EDA Notebook
Upload your dataset and this notebook will automatically clean it and perform EDA with insights.

In [ ]:
# Install required libraries (run once)
!pip install pandas numpy matplotlib seaborn missingno -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## Step 1 — Load Dataset
Change the file path below to your CSV file.

In [ ]:
# ✏️ Change this to your file path
FILE_PATH = 'your_dataset.csv'

df = pd.read_csv(FILE_PATH)
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head()

## Step 2 — Basic Info

In [ ]:
print('--- Shape ---')
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

print('\n--- Column Types ---')
print(df.dtypes)

print('\n--- Basic Info ---')
df.info()

## Step 3 — Data Cleaning

In [ ]:
# --- Duplicates ---
dups = df.duplicated().sum()
print(f'Duplicate rows: {dups}')
df = df.drop_duplicates()
print(f'After removing duplicates: {df.shape[0]} rows')

In [ ]:
# --- Missing Values ---
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct.round(2)})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if missing_df.empty:
    print('No missing values found!')
else:
    print('Columns with missing values:')
    print(missing_df)

In [ ]:
# --- Visualize Missing Values ---
if df.isnull().sum().sum() > 0:
    msno.matrix(df)
    plt.title('Missing Value Matrix')
    plt.show()
else:
    print('No missing values to visualize.')

In [ ]:
# --- Handle Missing Values ---
# Numeric columns: fill with median
num_cols = df.select_dtypes(include=np.number).columns.tolist()
for col in num_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].median(), inplace=True)
        print(f'Filled {col} with median')

# Categorical columns: fill with mode
cat_cols = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)
        print(f'Filled {col} with mode')

print(f'\nMissing values after cleaning: {df.isnull().sum().sum()}')

## Step 4 — Statistical Summary

In [ ]:
print('--- Numeric Columns Summary ---')
df[num_cols].describe().round(3)

In [ ]:
if cat_cols:
    print('--- Categorical Columns Summary ---')
    df[cat_cols].describe()

## Step 5 — Univariate Analysis

In [ ]:
# Histograms for numeric columns
if num_cols:
    df[num_cols].hist(bins=20, figsize=(15, len(num_cols) * 2), layout=(-1, 3), edgecolor='black')
    plt.suptitle('Distribution of Numeric Columns', y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Bar plots for categorical columns (top 10 values each)
for col in cat_cols:
    top = df[col].value_counts().head(10)
    top.plot(kind='bar', color='steelblue', edgecolor='black')
    plt.title(f'Top values in: {col}')
    plt.xlabel(col)
    plt.ylabel('Count')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## Step 6 — Outlier Detection

In [ ]:
# Boxplots for outlier detection
if num_cols:
    n = len(num_cols)
    cols_per_row = 3
    rows = (n + cols_per_row - 1) // cols_per_row
    fig, axes = plt.subplots(rows, cols_per_row, figsize=(15, rows * 4))
    axes = axes.flatten() if n > 1 else [axes]

    for i, col in enumerate(num_cols):
        axes[i].boxplot(df[col].dropna(), patch_artist=True,
                        boxprops=dict(facecolor='lightblue'))
        axes[i].set_title(col)

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('Boxplots — Outlier Detection', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# IQR-based outlier count per column
print('Outlier counts per numeric column (IQR method):')
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
    print(f'  {col}: {len(outliers)} outliers ({len(outliers)/len(df)*100:.1f}%)')

## Step 7 — Bivariate Analysis & Correlation

In [ ]:
# Correlation heatmap
if len(num_cols) >= 2:
    corr = df[num_cols].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, linewidths=0.5, square=True)
    plt.title('Correlation Heatmap')
    plt.tight_layout()
    plt.show()
else:
    print('Need at least 2 numeric columns for correlation.')

In [ ]:
# Top correlated pairs
if len(num_cols) >= 2:
    corr_pairs = corr.abs().unstack().sort_values(ascending=False)
    corr_pairs = corr_pairs[corr_pairs < 1].drop_duplicates()
    print('Top 10 most correlated column pairs:')
    print(corr_pairs.head(10).round(3))

In [ ]:
# Pairplot (for up to 5 numeric columns to keep it clean)
pairplot_cols = num_cols[:5]
if len(pairplot_cols) >= 2:
    sns.pairplot(df[pairplot_cols], diag_kind='kde', plot_kws={'alpha': 0.5})
    plt.suptitle('Pairplot of Numeric Features', y=1.01)
    plt.show()

## Step 8 — Insights Summary

In [ ]:
print('=' * 50)
print('           EDA INSIGHTS SUMMARY')
print('=' * 50)

print(f'\n📦 Dataset: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'   Numeric columns  : {len(num_cols)} → {num_cols}')
print(f'   Categorical cols : {len(cat_cols)} → {cat_cols}')

print(f'\n🧹 Data Quality:')
print(f'   Duplicates removed : {dups}')
remaining_missing = df.isnull().sum().sum()
print(f'   Missing values     : {remaining_missing} (after cleaning)')

print(f'\n📊 Numeric Statistics:')
for col in num_cols:
    print(f'   {col}: mean={df[col].mean():.2f}, std={df[col].std():.2f}, min={df[col].min():.2f}, max={df[col].max():.2f}')

print(f'\n🏷️  Categorical Summary:')
for col in cat_cols:
    print(f'   {col}: {df[col].nunique()} unique values, most common = "{df[col].mode()[0]}" ({df[col].value_counts().iloc[0]} times)')

if len(num_cols) >= 2:
    top_pair = corr_pairs.idxmax()
    top_val = corr_pairs.max()
    print(f'\n🔗 Strongest Correlation: {top_pair[0]} ↔ {top_pair[1]} (r = {top_val:.3f})')

print(f'\n⚠️  Outlier Alert:')
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    out_count = len(df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)])
    if out_count > 0:
        print(f'   {col}: {out_count} outliers detected')

print('\n' + '=' * 50)
print('✅ EDA Complete. Dataset is ready for modeling.')
print('=' * 50)

In [ ]:
# Save cleaned dataset
df.to_csv('cleaned_dataset.csv', index=False)
print('Cleaned dataset saved as cleaned_dataset.csv')